### Task 1 c)

In [150]:
import random
import matplotlib.pyplot as plt
import numpy as np
from fractions import Fraction
import pandas as pd

In [151]:
#Probabilities
alfa = 0.005
beta = 0.01
gamma = 0.1

X0 = 0

#A function for "randomly" finding the state of the individual the nest day
def one_day(X_curr):

    p = random.uniform(0, 1)

    if X_curr == 0:
        if p <= beta:
            return 1
        else:
            return X_curr

    elif X_curr == 1:
        if p <= gamma:
            return 2
        else: 
            return X_curr
    
    elif X_curr == 2:
        if p <= alfa:
            return 0
        else:
            return X_curr

#Simulating the Markov chain for 20 years = 7300 time steps

def twenty_years():
    state_history = []
    state_history.append(X0)
    X_curr = X0
    
    for i in range(1, 7300):
        X_next = one_day(X_curr)
        state_history.append(X_next)
        X_curr = X_next
    
    last_ten_years = state_history[3650:]

    num_of_days = [(last_ten_years.count(0))/10, (last_ten_years.count(1))/10, (last_ten_years.count(2))/10]
    return num_of_days

states = ['S', 'I', 'R']

"""#Plotting the limiting distributon after 10 years
plt.figure(figsize=(6, 4))
plt.bar(states, num_of_days, color = 'mediumorchid')
#labels = [str(Fraction(num).limit_denominator()) for num in num_of_days]
plt.bar_label(plt.bar(states, num_of_days, color = 'mediumorchid'))
plt.title('State distribution')
plt.xlabel('States')
plt.ylabel('State history')
plt.show()"""



"#Plotting the limiting distributon after 10 years\nplt.figure(figsize=(6, 4))\nplt.bar(states, num_of_days, color = 'mediumorchid')\n#labels = [str(Fraction(num).limit_denominator()) for num in num_of_days]\nplt.bar_label(plt.bar(states, num_of_days, color = 'mediumorchid'))\nplt.title('State distribution')\nplt.xlabel('States')\nplt.ylabel('State history')\nplt.show()"

In [152]:
#30 years + approximating the 95% confidence inetrval
states = ['S', 'I', 'R']
results = []
for i in range(1, 31):
    results.append(twenty_years())


df = pd.DataFrame(results, columns=states, index = range(1, 31))

df["Total"] = df.sum(axis=1)

display(
    df.style
    .set_caption("Results from 30 simulations")
    .format("{:.0f}")
    .background_gradient(subset=states, cmap="Blues")
)



,S,I,R,Total
1,57,8,300,365
2,80,10,274,365
3,139,9,217,365
4,168,10,187,365
5,56,11,298,365
6,130,16,219,365
7,79,11,275,365
8,131,7,227,365
9,138,7,220,365
10,110,12,242,365


In [153]:
n = 30
t_value = 2.045

#Using inly the state columns-not total
state_data = df[states]

def CI(state: int):   
    x_vals = []
    results = []
    for i in range(1, 31):
        results.append(twenty_years())
        
    for i in range(n):
        x_vals.append(results[i][state])

    mean = np.mean(x_vals)

    s = np.sqrt(1/(n-1)*sum((mean-x_vals)**2))

    SE = np.sqrt(s**2/n)

    lower = mean - t_value*SE
    upper = mean + t_value*SE

    return np.array([lower, upper])

#Computing the confidence interval for every state
CI_S = CI(0)
CI_I = CI(1)
CI_R = CI(2)


#Plotting a table for the CI's
ci_df = pd.DataFrame({
    "S" : CI_S, 
    "I" : CI_I, 
    "R" : CI_R},
    index = ["Lower", "Upper"]
     
    )

display(
    ci_df.style
    .set_caption("95% Confidence Intervals")
    .set_table_styles([
        {
            "selector":"caption",
            "props" : [("font-size", "20px")]
        }
    ])
    .format("{:2f}")
    .set_properties(**{
        "font-size": "16px",
        "min-width": "120px",
        "padding" : "10px"
    })
    .background_gradient(cmap = "viridis", axis = None)
)


,S,I,R
Lower,112.702238,10.292761,221.544721
Upper,130.717762,13.600572,246.235279


In [154]:
#Checking how many times the theoretical long-run number of days per year is inside the interval
#theoretical values
pi_0 = 117.74
pi_1 = 11.774
pi_2 = 235.48

s_count = 0
i_count = 0
r_count = 0
for j in range(1000):
    

    ci_s = CI(0)
    ci_i = CI(1)
    ci_r = CI(2)

    if ci_s[0] <= pi_0 and ci_s[1] >= pi_0:
        s_count += 1
    if ci_i[0] <= pi_1 and ci_i[1] >= pi_1:
            i_count += 1
    if ci_r[0] <= pi_2 and ci_r[1] >= pi_2:
            r_count += 1


print(s_count/1000)
print(i_count/1000)
print(r_count/1000)



0.953
0.952
0.954
